# Lezione 11 — Modelli generativi moderni: autoencoder variazionali

Nella lezione dieci hai costruito un autoencoder che comprime un'immagine in un vettore e la ricostruisce. In questa lezione fai il passo successivo: trasformi quel vettore fisso in una distribuzione di probabilità e ottieni un vero modello generativo, l'autoencoder variazionale (VAE, dall'inglese *Variational Autoencoder*). Esegui ogni cella in ordine, su Google Colab, e vedi il modello addestrarsi e generare cifre nuove dal vivo.

## Obiettivi di apprendimento

- Spiegare la differenza tra lo spazio latente di un autoencoder normale e quello di un autoencoder variazionale
- Implementare il trucco della riparametrizzazione per campionare da una gaussiana in modo differenziabile
- Costruire e addestrare un piccolo VAE sul dataset di cifre manoscritte MNIST
- Calcolare la funzione di perdita ELBO (limite inferiore dell'evidenza, *Evidence Lower Bound*), che combina ricostruzione e divergenza di Kullback-Leibler (KL)
- Generare cifre nuove campionando punti a caso dallo spazio latente e decodificandoli

## Da un punto a una distribuzione

L'autoencoder della lezione precedente assegna a ogni immagine un unico punto fisso nello spazio latente. Quel punto non ti garantisce nulla sui suoi vicini: due immagini simili possono finire lontane tra loro, e un punto intermedio tra due codifiche può non significare nulla quando lo decodifichi.

L'autoencoder variazionale cambia la regola. Invece di un punto, il codificatore produce i parametri di una distribuzione gaussiana: una media (mu) e una varianza (sigma al quadrato) per ogni immagine. L'addestramento spinge inoltre tutte queste distribuzioni a somigliare a una gaussiana standard centrata in zero, il che ordina lo spazio latente ed elimina i vuoti. Il risultato è una mappa continua: qualunque punto tu scelga, che sia o no vicino a un dato di addestramento, si decodifica in qualcosa di sensato.

## Il trucco della riparametrizzazione

Per addestrare il codificatore con la discesa del gradiente, devi propagare l'errore attraverso il passo di campionamento. Campionare direttamente da una gaussiana con media e varianza apprese blocca il gradiente: non esiste una derivata ben definita di "un numero scelto a caso" rispetto ai parametri che definiscono quel caso.

La soluzione separa il caso dai parametri addestrabili. Generi un rumore fisso (epsilon) da una gaussiana standard, che non dipende da alcun parametro, e lo combini con mu e sigma tramite una formula differenziabile:

z = mu + sigma · epsilon

Il gradiente attraversa questa formula senza problemi, perché è una semplice moltiplicazione e una somma. Il caso resta isolato, fuori dal percorso dell'apprendimento.

In [ ]:
import torch

# mu y log-varianza, como si vinieran de la salida de un codificador
mu = torch.tensor([0.5, -1.2], requires_grad=True)
logvar = torch.tensor([0.1, -0.3], requires_grad=True)

# sigma siempre positiva, aunque logvar pueda ser cualquier número real
sigma = torch.exp(0.5 * logvar)

# epsilon: ruido fijo, sin gradiente, generado aparte de mu y sigma
epsilon = torch.randn_like(sigma)

# truco de reparametrización: z es una función diferenciable de mu y sigma
z = mu + sigma * epsilon
print("z:", z)

# comprueba que el gradiente fluye hasta mu y logvar
z.sum().backward()
print("gradiente respecto a mu:", mu.grad)
print("gradiente respecto a logvar:", logvar.grad)

## Costruisci un VAE

Ora unisci i due elementi in un modello completo. Il codificatore prende un'immagine di 28 per 28 pixel del dataset MNIST e produce mu e log-varianza. Il campionamento con riparametrizzazione ottiene un punto z dello spazio latente. Il decodificatore ricostruisce l'immagine a partire da z. Addestri il modello con la perdita ELBO: ricostruzione più divergenza KL rispetto al prior.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# MNIST: dígitos manuscritos de 28x28 píxeles en escala de grises, valores en [0, 1]
transform = transforms.ToTensor()
train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

LATENT_DIM = 20  # dimensión del espacio latente z

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=LATENT_DIM):
        super().__init__()
        # codificador: capa oculta compartida
        self.encoder_hidden = nn.Linear(input_dim, hidden_dim)
        # dos cabezas separadas: una produce mu, otra produce log-varianza
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # decodificador: del espacio latente de vuelta a los píxeles
        self.decoder_hidden = nn.Linear(latent_dim, hidden_dim)
        self.decoder_out = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        h = F.relu(self.encoder_hidden(x))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        # z = mu + sigma * epsilon, con epsilon ~ N(0, I)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        h = F.relu(self.decoder_hidden(z))
        # sigmoid: el decodificador modela una Bernoulli por píxel
        return torch.sigmoid(self.decoder_out(h))

    def forward(self, x):
        mu, logvar = self.encode(x.view(x.size(0), -1))
        z = self.reparameterize(mu, logvar)
        x_reconstruida = self.decode(z)
        return x_reconstruida, mu, logvar


modelo = VAE().to(device)
print(modelo)

In [ ]:
def elbo_loss(x_reconstruida, x, mu, logvar):
    # término de reconstrucción: log-verosimilitud de una Bernoulli, sumada sobre píxeles
    reconstruccion = F.binary_cross_entropy(
        x_reconstruida, x.view(x.size(0), -1), reduction="sum"
    )
    # divergencia KL entre q(z|x) = N(mu, sigma^2) y el prior p(z) = N(0, I), forma cerrada
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return reconstruccion + kl  # minimizar esto equivale a maximizar el ELBO

In [ ]:
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)

EPOCHS = 10
for epoch in range(1, EPOCHS + 1):
    modelo.train()
    perdida_total = 0.0
    for x, _ in train_loader:  # las etiquetas no se usan: el VAE es no supervisado
        x = x.to(device)
        optimizador.zero_grad()
        x_reconstruida, mu, logvar = modelo(x)
        perdida = elbo_loss(x_reconstruida, x, mu, logvar)
        perdida.backward()
        optimizador.step()
        perdida_total += perdida.item()

    print(f"Época {epoch}/{EPOCHS} — pérdida media: {perdida_total / len(train_data):.2f}")

In [ ]:
# comprobación rápida: ¿reconstruye bien el modelo dígitos que no vio como test?
modelo.eval()
with torch.no_grad():
    x_test, _ = next(iter(test_loader))
    x_test = x_test.to(device)
    x_reconstruida, _, _ = modelo(x_test)

n = 8
fig, ejes = plt.subplots(2, n, figsize=(n * 1.2, 3))
for i in range(n):
    ejes[0, i].imshow(x_test[i].cpu().squeeze(), cmap="gray")
    ejes[0, i].axis("off")
    ejes[1, i].imshow(x_reconstruida[i].cpu().view(28, 28), cmap="gray")
    ejes[1, i].axis("off")
plt.suptitle("Fila superior: dígitos originales — fila inferior: reconstrucciones del VAE")
plt.show()

## Genera cifre nuove

Il decodificatore sa già convertire un punto dello spazio latente in un'immagine. Ora fai a meno del tutto del codificatore: prendi punti a caso direttamente dal prior, la gaussiana standard N(0, I), e li passi al decodificatore. Se lo spazio latente è stato ben ordinato durante l'addestramento, ogni punto produce una cifra riconoscibile, senza che alcuna immagine reale sia intervenuta nel processo.

In [ ]:
modelo.eval()
with torch.no_grad():
    n_muestras = 64
    z_aleatorio = torch.randn(n_muestras, LATENT_DIM).to(device)  # muestreo directo del prior N(0, I)
    muestras = modelo.decode(z_aleatorio).cpu()

filas, columnas = 8, 8
fig, ejes = plt.subplots(filas, columnas, figsize=(columnas * 1.2, filas * 1.2))
for i, ax in enumerate(ejes.flat):
    ax.imshow(muestras[i].view(28, 28), cmap="gray")
    ax.axis("off")
plt.suptitle("Dígitos generados desde puntos aleatorios del espacio latente")
plt.show()

## Riepilogo della lezione

- Un autoencoder normale assegna a ogni dato un punto fisso nello spazio latente; un VAE gli assegna una distribuzione di probabilità, e questo trasforma lo spazio latente in una mappa continua
- Il trucco della riparametrizzazione separa il caso (epsilon) dai parametri addestrabili (mu e sigma), permettendo di propagare i gradienti attraverso il campionamento: z = mu + sigma · epsilon
- Il codificatore di un VAE produce due vettori, mu e log-varianza, invece di un unico vettore di codice
- La perdita ELBO combina un termine di ricostruzione (quanto bene il decodificatore ricostruisce l'input) e un termine di divergenza KL (quanto la distribuzione del codificatore è vicina al prior gaussiano)
- Generare dati nuovi è semplice quanto campionare dal prior N(0, I) e decodificare, senza bisogno di partire da alcuna immagine reale

**Apri ora il notebook di pratica di accompagnamento, `../notebooks_practica/clase11_practica_modelos_generativos_modernos.ipynb`, e sperimenta con lo spazio latente per conto tuo.**